# R2Gen Baseline — Google Colab

**Project:** Vision-Language Models in Radiology  
**Model:** R2Gen — Memory-Driven Transformer for radiology report generation (Chen et al., EMNLP 2020)

This notebook:
1. Sets up the environment
2. Downloads the IU X-Ray dataset
3. Builds the report vocabulary
4. Runs architecture smoke tests
5. Trains R2Gen
6. Evaluates with BLEU, ROUGE-L, METEOR, and proxy RadGraph F1

> **Runtime:** Set *Runtime → Change runtime type → T4 GPU* before running.

---
## 1. Environment Setup

In [ ]:
import os, sys

REPO_URL = "https://github.com/SinaDns/radiology-vision-language-models.git"
REPO_DIR = "/content/radiology-vision-language-models"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    datefmt='%H:%M:%S'
)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

---
## 2. Download IU X-Ray Dataset

In [ ]:
from pathlib import Path
import tarfile

DATA_DIR    = Path("/content/radiology-vision-language-models/data/iu_xray")
IMAGES_DIR  = DATA_DIR / "images"
REPORTS_DIR = DATA_DIR / "reports"
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

IMAGE_TGZ  = DATA_DIR / "NLMCXR_png.tgz"
REPORT_TGZ = DATA_DIR / "NLMCXR_reports.tgz"

if not IMAGE_TGZ.exists():
    print("Downloading images (~1.3 GB)…")
    !wget -q --show-progress \
        "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz" \
        -O {IMAGE_TGZ}

if not REPORT_TGZ.exists():
    print("Downloading reports…")
    !wget -q --show-progress \
        "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz" \
        -O {REPORT_TGZ}

if not any(IMAGES_DIR.glob("*.png")):
    print("Extracting images…")
    with tarfile.open(IMAGE_TGZ, "r:gz") as tar:
        for m in tar.getmembers():
            m.name = Path(m.name).name
            tar.extract(m, IMAGES_DIR)

if not any(REPORTS_DIR.glob("*.xml")):
    print("Extracting reports…")
    with tarfile.open(REPORT_TGZ, "r:gz") as tar:
        for m in tar.getmembers():
            m.name = Path(m.name).name
            tar.extract(m, REPORTS_DIR)

print(f"PNG images : {len(list(IMAGES_DIR.glob('*.png')))}")
print(f"XML reports: {len(list(REPORTS_DIR.glob('*.xml')))}")

---
## 3. Build Vocabulary

In [ ]:
import os
os.makedirs("experiments/results", exist_ok=True)

from src.data_loaders.iu_xray_seq2seq import build_tokenizer

VOCAB_PATH = "experiments/results/r2gen_vocab.json"
tokenizer = build_tokenizer(
    data_dir="data/iu_xray/",
    save_path=VOCAB_PATH,
    min_freq=3,
)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens — PAD={tokenizer.pad_id} UNK={tokenizer.unk_id} "
      f"BOS={tokenizer.bos_id} EOS={tokenizer.eos_id}")

# Sanity check
sample = "there is no acute cardiopulmonary disease"
ids = tokenizer.encode(sample)
decoded = tokenizer.decode(ids)
print(f"\nEncode: '{sample}'")
print(f"IDs ({len(ids)}): {ids[:10]}…")
print(f"Decoded: '{decoded}'")

---
## 4. Architecture Smoke Tests

In [ ]:
import torch
from src.models.r2gen import R2GenModel

# This downloads ResNet-101 ImageNet weights on first run (~170 MB)
model = R2GenModel(
    vocab_size=tokenizer.vocab_size,
    d_model=512, num_heads=8,
    num_enc_layers=3, num_dec_layers=3,
    dim_ff=2048, dropout=0.1,
    num_mem_slots=3, max_seq_len=100,
    pretrained_image=True,
    pad_id=tokenizer.pad_id,
)

B = 2
imgs     = torch.randn(B, 3, 224, 224)
inp_ids  = torch.randint(0, tokenizer.vocab_size, (B, 20))
logits   = model(imgs, inp_ids)
print(f"Forward pass OK — logits shape: {logits.shape}")
# Expected: (2, 20, vocab_size)

# Beam search test
with torch.no_grad():
    seqs = model.generate(imgs, bos_id=tokenizer.bos_id,
                          eos_id=tokenizer.eos_id, beam_size=2, max_length=15)
print(f"Beam search OK — generated {len(seqs)} sequences")
print(f"Sample: {tokenizer.decode(seqs[0])}")

In [ ]:
# Factuality loss smoke test
import torch
from src.training.factuality_loss import ProxyFactualityLoss

B, T, V = 4, 20, tokenizer.vocab_size
logits  = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))

crit = ProxyFactualityLoss(vocab=tokenizer.word2idx, coverage_weight=0.3)
total, ce, fact = crit(logits, targets)
print(f"ProxyFactualityLoss OK — total={total:.4f} ce={ce:.4f} fact={fact:.4f}")

---
## 5. Training

In [ ]:
from src.utils.config import load_config

config = load_config("experiments/configs/r2gen.yaml")

# Colab-specific overrides
config["paths"]["iu_xray_dir"]    = "data/iu_xray/"
config["paths"]["checkpoint_dir"] = "experiments/results/checkpoints/r2gen/"
config["paths"]["log_dir"]        = "experiments/results/logs/"
config["tokenizer"]["vocab_save_path"] = VOCAB_PATH
config["training"]["batch_size"]   = 8     # T4-safe; use 16 for A100
config["training"]["epochs"]       = 10    # ~35 min on T4; increase to 30 for full training
config["data"]["num_workers"]      = 2
print("Config ready.")

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.data_loaders.iu_xray_seq2seq import IUXraySeq2SeqDataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
from src.models.r2gen import R2GenModel
from src.training.r2gen_trainer import R2GenTrainer
from src.utils.logging_utils import setup_logger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

image_size  = config["data"]["image_size"]
max_length  = config["tokenizer"]["max_length"]
iu_dir      = config["paths"]["iu_xray_dir"]
val_frac    = config["data"]["val_split"]

train_ds = IUXraySeq2SeqDataset(
    data_dir=iu_dir, tokenizer=tokenizer, split="train",
    val_fraction=val_frac, transform=get_train_transforms(image_size),
    max_length=max_length,
)
val_ds = IUXraySeq2SeqDataset(
    data_dir=iu_dir, tokenizer=tokenizer, split="val",
    val_fraction=val_frac, transform=get_val_transforms(image_size),
    max_length=max_length,
)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

bs = config["training"]["batch_size"]
nw = config["data"]["num_workers"]

train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                          num_workers=nw, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                          num_workers=nw, pin_memory=True)

model_cfg = config["model"]
model = R2GenModel(
    vocab_size=tokenizer.vocab_size,
    d_model=model_cfg["d_model"], num_heads=model_cfg["num_heads"],
    num_enc_layers=model_cfg["num_enc_layers"],
    num_dec_layers=model_cfg["num_dec_layers"],
    dim_ff=model_cfg["dim_ff"], dropout=model_cfg["dropout"],
    num_mem_slots=model_cfg["num_mem_slots"],
    max_seq_len=model_cfg["max_seq_len"],
    pretrained_image=model_cfg.get("pretrained_image", True),
    pad_id=tokenizer.pad_id,
)

logger = setup_logger(config["paths"]["log_dir"], config["logging"]["run_name"])
trainer = R2GenTrainer(
    model=model, config=config,
    train_loader=train_loader, val_loader=val_loader,
    device=device, pad_id=tokenizer.pad_id, logger=logger,
)
print("Trainer ready.")

In [ ]:
# ── Run training ────────────────────────────────────────────────────────────
trainer.train(resume_path=None)

In [ ]:
# Save best checkpoint to Drive (optional)
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy("experiments/results/checkpoints/r2gen/best.pt",
#             "/content/drive/MyDrive/r2gen_best.pt")

---
## 6. Evaluation: BLEU / ROUGE-L / METEOR / Proxy RadGraph F1

In [ ]:
import torch
import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

from torch.utils.data import DataLoader
from src.data_loaders.iu_xray_seq2seq import IUXraySeq2SeqDataset
from src.data_loaders.transforms import get_val_transforms
from src.evaluation.generation_metrics import compute_all_metrics, generation_report
from src.training.factuality_loss import compute_radgraph_f1

CHECKPOINT = "experiments/results/checkpoints/r2gen/best.pt"
# CHECKPOINT = "/content/drive/MyDrive/r2gen_best.pt"  # from Drive

ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
print(f"Loaded checkpoint (epoch {ckpt.get('epoch', '?')}).")

# Generate on val set
val_ds_eval = IUXraySeq2SeqDataset(
    data_dir=iu_dir, tokenizer=tokenizer, split="val",
    val_fraction=val_frac, transform=get_val_transforms(image_size),
    max_length=max_length,
)
val_loader_eval = DataLoader(val_ds_eval, batch_size=8, shuffle=False,
                             num_workers=2, pin_memory=True)

hypotheses, references = [], []

for batch_idx, batch in enumerate(val_loader_eval):
    imgs = batch["image"].to(device)
    refs = batch["report"]
    seqs = model.generate(
        imgs, bos_id=tokenizer.bos_id, eos_id=tokenizer.eos_id,
        beam_size=3, max_length=100,
    )
    for s in seqs:
        hypotheses.append(tokenizer.decode(s))
    references.extend(refs)
    if (batch_idx + 1) % 20 == 0:
        print(f"Generated {batch_idx+1}/{len(val_loader_eval)} batches")

print(f"\nTotal generated: {len(hypotheses)}")
print("\nSample output:")
for i in range(min(3, len(hypotheses))):
    print(f"  Ref  [{i}]: {references[i][:120]}")
    print(f"  Hyp  [{i}]: {hypotheses[i][:120]}")
    print()

In [ ]:
# Standard generation metrics
metrics = compute_all_metrics(hypotheses, references)
print("\n=== R2Gen Generation Metrics (IU X-Ray val) ===")
print(generation_report(metrics))

# Proxy RadGraph F1
rg_metrics = compute_radgraph_f1(hypotheses, references)
for k, v in rg_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTarget baselines: BLEU-4 > 0.10, ROUGE-L > 0.30")